# Week 45, Day 3: Decisions as attractor dynamics

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w045/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Psychometric and chronometric functions

Complete one Euler step of the model inside sim(): compute x1 and x2, the rates H(x1) and H(x2), and update S1, S2 and the two noise currents.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
a_, b_, d_ = 270.0, 108.0, 0.154
gamma, J11, J12, I0, JA, mu0, tauA, sigma = 0.641, 0.2609, 0.0497, 0.3255, 5.2e-4, 30.0, 0.002, 0.02

def H(x):
    return (a_ * x - b_) / (1 - np.exp(-d_ * (a_ * x - b_)))

def sim(c, n=300, T=1.5, dt=0.0005, thr=15.0, tauS=0.1, stim=(0.0, None), seed=0):
    g = np.random.default_rng(seed)
    S1 = np.full(n, 0.1); S2 = np.full(n, 0.1); N1 = np.zeros(n); N2 = np.zeros(n)
    ch = np.full(n, -1); rt = np.full(n, np.nan)
    for k in range(int(T / dt)):
        t = k * dt
        on = float(t >= stim[0] and (stim[1] is None or t < stim[1]))
        I1, I2 = JA * mu0 * (1 + c / 100) * on, JA * mu0 * (1 - c / 100) * on
        ...
        new = (ch < 0) & ((r1 > thr) | (r2 > thr))
        ch[new] = np.where(r1[new] > r2[new], 1, 0); rt[new] = t
    return ch, rt, r1, r2

cohs = [0.0, 3.2, 6.4, 12.8, 25.6, 51.2]
res = {}
for c in cohs:
    ch, rt, _, _ = sim(c)
    dec = ch >= 0
    res[c] = ((ch[dec] == 1).mean(), np.nanmean(rt))
    print(f"coherence {c:4.1f}%: decided {dec.mean():.2f}, accuracy {res[c][0]:.3f}, mean decision time {res[c][1]:.3f} s")
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].semilogx([max(c, 1) for c in cohs], [res[c][0] for c in cohs], "o-"); ax[0].set_ylabel("accuracy")
ax[1].semilogx([max(c, 1) for c in cohs], [res[c][1] for c in cohs], "o-"); ax[1].set_ylabel("decision time (s)")
for a in ax: a.set_xlabel("coherence (%)")
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
def test_psych():
    "accuracy rises with coherence from about chance to near perfect"
    acc = [res[c][0] for c in cohs]
    assert abs(acc[0] - 0.5) < 0.08 and acc[-1] > 0.97 and all(np.diff(acc) > -0.02)
def test_chrono():
    "decision time falls with coherence"
    t = [res[c][1] for c in cohs]
    assert t[0] > t[-1] + 0.1

test_psych()
test_chrono()
print("All checks passed")

### Exercise: Activity after the stimulus ends

Run 200 trials with 12.8% coherence shown only for the first second of a 3-second trial, and 200 trials with no stimulus at all. Report the final rates.

In [ ]:
_, _, r1m, r2m = ...
_, _, r1s, r2s = ...
print(f"stimulus off for 2 s: winner {np.mean(np.maximum(r1m, r2m)):.1f} Hz, loser {np.mean(np.minimum(r1m, r2m)):.1f} Hz")
print(f"no stimulus: populations at {np.mean(r1s):.1f} and {np.mean(r2s):.1f} Hz")

In [ ]:
import numpy as np
def test_persist():
    "two seconds after the stimulus ends, the winning population fires above 15 Hz and the loser below 2 Hz"
    assert np.mean(np.maximum(r1m, r2m)) > 15 and np.mean(np.minimum(r1m, r2m)) < 2
def test_spont():
    "without any stimulus, neither population leaves its spontaneous state"
    assert np.mean(r1s) < 5 and np.mean(r2s) < 5

test_persist()
test_spont()
print("All checks passed")

### Exercise: Fast synapses

Run 400 trials at 6.4% coherence with tauS = 0.1 and with tauS = 0.02, and report the fraction of trials that reach the 15 Hz threshold.

In [ ]:
ch_s, _, _, _ = ...
ch_f, _, _, _ = ...
dec_slow, dec_fast = (ch_s >= 0).mean(), (ch_f >= 0).mean()
print(f"decided: slow synapses {dec_slow:.2f}, fast synapses {dec_fast:.2f}")

In [ ]:
import numpy as np
def test_fast():
    "with a 20 ms synaptic time constant, the network almost never reaches a decision at 6.4% coherence"
    assert dec_fast < 0.05 and dec_slow > 0.95

test_fast()
print("All checks passed")